# Big Data Systems Project on Amazon Product Review Analysis using Apache Spark

The Amazon Product Reviews dataset is analyzed using PySpark.

In [1]:
!pip install pyspark


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviewAnalysis") \
    .getOrCreate()

print("Spark Session Created Successfully")
# Print Spark Version
print("Spark Version:", spark.version)


Spark Session Created Successfully
Spark Version: 4.0.2


## Data Loading

The dataset is loaded into Spark using proper CSV parsing options.

Since the file contains:
- multi-line reviews  
- commas inside text  
- embedded quotes  

special options such as multiLine, escape and quote are used to correctly parse the data.

After loading, schema and record count are displayed.


In [3]:
df = spark.read \
    .option("header", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .option("quote", "\"") \
    .option("inferSchema", "true") \
    .csv("AmazonProductReviews.csv")

df.printSchema()

print("Total Records Loaded:", df.count())


root
 |-- id: string (nullable = true)
 |-- asins: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- colors: string (nullable = true)
 |-- dateAdded: timestamp (nullable = true)
 |-- dateUpdated: timestamp (nullable = true)
 |-- dimension: string (nullable = true)
 |-- ean: double (nullable = true)
 |-- keys: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- manufacturerNumber: string (nullable = true)
 |-- name: string (nullable = true)
 |-- prices: string (nullable = true)
 |-- reviews.date: timestamp (nullable = true)
 |-- reviews.doRecommend: boolean (nullable = true)
 |-- reviews.numHelpful: integer (nullable = true)
 |-- reviews.rating: integer (nullable = true)
 |-- reviews.sourceURLs: string (nullable = true)
 |-- reviews.text: string (nullable = true)
 |-- reviews.title: string (nullable = true)
 |-- reviews.userCity: string (nullable = true)
 |-- reviews.userProvince: string (nullable = true)
 

## Column Renaming

Several columns contain dots in their names (e.g., reviews.rating).  
Spark interprets these as nested fields, causing query issues.

Therefore these columns are renamed to simple names for easier processing.


In [4]:
df = df.withColumnRenamed("reviews.rating", "reviews_rating") \
       .withColumnRenamed("reviews.text", "reviews_text") \
       .withColumnRenamed("reviews.title", "reviews_title") \
       .withColumnRenamed("reviews.date", "reviews_date") \
       .withColumnRenamed("reviews.username", "reviews_username")

print("Columns Renamed Successfully")
df.printSchema()


Columns Renamed Successfully
root
 |-- id: string (nullable = true)
 |-- asins: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- colors: string (nullable = true)
 |-- dateAdded: timestamp (nullable = true)
 |-- dateUpdated: timestamp (nullable = true)
 |-- dimension: string (nullable = true)
 |-- ean: double (nullable = true)
 |-- keys: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- manufacturerNumber: string (nullable = true)
 |-- name: string (nullable = true)
 |-- prices: string (nullable = true)
 |-- reviews_date: timestamp (nullable = true)
 |-- reviews.doRecommend: boolean (nullable = true)
 |-- reviews.numHelpful: integer (nullable = true)
 |-- reviews_rating: integer (nullable = true)
 |-- reviews.sourceURLs: string (nullable = true)
 |-- reviews_text: string (nullable = true)
 |-- reviews_title: string (nullable = true)
 |-- reviews.userCity: string (nullable = true)
 |-- reviews.userProvinc

## Data Cleansing

This step performs required cleaning:

1. Create new column primary_category from categories  
2. Convert rating column to integer  
3. Remove rows where rating is missing or outside 1–5  

The cleaned schema and record count are printed.


In [5]:
from pyspark.sql.functions import split, col

# Create primary category column
from pyspark.sql.functions import trim

df_clean = df.withColumn(
    "primary_category",
    trim(split(col("categories"), ",").getItem(0))
)

# Convert rating to integer
df_clean = df_clean.withColumn("reviews_rating_int", col("reviews_rating").cast("int"))

# Keep only valid ratings 1 to 5
df_clean = df_clean.filter(
    (col("reviews_rating_int").isNotNull()) &
    (col("reviews_rating_int") >= 1) &
    (col("reviews_rating_int") <= 5)
)

df_clean.printSchema()

print("Records after cleaning:", df_clean.count())


root
 |-- id: string (nullable = true)
 |-- asins: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- colors: string (nullable = true)
 |-- dateAdded: timestamp (nullable = true)
 |-- dateUpdated: timestamp (nullable = true)
 |-- dimension: string (nullable = true)
 |-- ean: double (nullable = true)
 |-- keys: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- manufacturerNumber: string (nullable = true)
 |-- name: string (nullable = true)
 |-- prices: string (nullable = true)
 |-- reviews_date: timestamp (nullable = true)
 |-- reviews.doRecommend: boolean (nullable = true)
 |-- reviews.numHelpful: integer (nullable = true)
 |-- reviews_rating: integer (nullable = true)
 |-- reviews.sourceURLs: string (nullable = true)
 |-- reviews_text: string (nullable = true)
 |-- reviews_title: string (nullable = true)
 |-- reviews.userCity: string (nullable = true)
 |-- reviews.userProvince: string (nullable = true)
 

## Top Products by Average Rating with Minimum Reviews

Products having at least 20 reviews are selected.  
They are ranked by their average rating.

This helps identify highly rated and reliable products.


In [6]:
from pyspark.sql.functions import avg, count

top_products = df_clean.groupBy("name") \
    .agg(
        avg("reviews_rating_int").alias("avg_rating"),
        count("reviews_rating_int").alias("review_count")
    ) \
    .filter(col("review_count") >= 20) \
    .orderBy(col("avg_rating").desc())

top_products.show(20, truncate=False)


+-----------------------------------------------------+------------------+------------+
|name                                                 |avg_rating        |review_count|
+-----------------------------------------------------+------------------+------------+
|Fire HD 6 Tablet                                     |5.0               |38          |
|Kindle Paperwhite                                    |4.590909090909091 |22          |
|Amazon Tap - Alexa-Enabled Portable Bluetooth Speaker|4.533210332103321 |542         |
|Kindle Fire HDX 7"                                   |4.391304347826087 |23          |
|Amazon Fire TV                                       |4.204545454545454 |44          |
|Amazon Premium Headphones                            |4.012987012987013 |77          |
|All-New Amazon Fire 7 Tablet Case (7th Generation    |3.7777777777777777|27          |
+-----------------------------------------------------+------------------+------------+



## Most Active Reviewers

This query lists top 10 reviewers based on number of reviews written.

It helps analyze user engagement levels.


In [7]:
active_reviewers = df_clean.groupBy("reviews_username") \
    .count() \
    .orderBy(col("count").desc())

active_reviewers.show(10, truncate=False)


+----------------+-----+
|reviews_username|count|
+----------------+-----+
|A. Younan       |38   |
|Andrew          |23   |
|William Hardin  |23   |
|Amazon Customer |17   |
|Victor L.       |15   |
|NF              |13   |
|Earthling1984   |13   |
|Amazon Reviewer |12   |
|Mike W.         |12   |
|J. Chambers     |10   |
+----------------+-----+
only showing top 10 rows


## Monthly Trend of Average Ratings per Category

This query analyzes how ratings change over time for each primary category.

Reviews are grouped by month and category and average ratings are computed.


In [8]:
from pyspark.sql.functions import to_date, date_format

from pyspark.sql.functions import to_timestamp

trend = df_clean.withColumn(
    "month",
    date_format(to_timestamp("reviews_date"), "yyyy-MM")
)

trend_result = trend.groupBy("primary_category", "month") \
    .agg(avg("reviews_rating_int").alias("avg_rating")) \
    .orderBy("month")

trend_result.show(40, truncate=False)


+----------------------------+-------+------------------+
|primary_category            |month  |avg_rating        |
+----------------------------+-------+------------------+
|Categories                  |NULL   |5.0               |
|Electronics                 |NULL   |4.906976744186046 |
|Cell Phones & Accessories   |NULL   |3.6666666666666665|
|Amazon Devices              |NULL   |4.148936170212766 |
|Kindle Store                |NULL   |4.0               |
|Amazon Devices & Accessories|NULL   |3.0625            |
|Amazon Devices              |2012-09|4.5               |
|Amazon Devices              |2012-10|4.0               |
|Amazon Devices              |2013-10|4.3125            |
|Amazon Devices              |2013-11|4.166666666666667 |
|Amazon Devices              |2013-12|5.0               |
|Categories                  |2014-04|4.24              |
|Categories                  |2014-05|5.0               |
|Categories                  |2014-06|1.0               |
|Amazon Device

## Products Loved by Some and Hated by Few

For each product the ratio of 5-star to 1-star reviews is computed.

Products with high ratio are mostly loved and rarely disliked.


In [9]:
from pyspark.sql.functions import sum as spark_sum, when

ratio_df = df_clean.groupBy("name").agg(
    spark_sum(when(col("reviews_rating_int") == 5, 1).otherwise(0)).alias("five_star"),
    spark_sum(when(col("reviews_rating_int") == 1, 1).otherwise(0)).alias("one_star")
)

ratio_df = ratio_df.filter(col("one_star") > 0)

ratio_df = ratio_df.withColumn(
    "ratio",
    col("five_star") / col("one_star")
)

ratio_df.orderBy(col("ratio").desc()).show(10, truncate=False)


+-----------------------------------------------------------------------------------------+---------+--------+------------------+
|name                                                                                     |five_star|one_star|ratio             |
+-----------------------------------------------------------------------------------------+---------+--------+------------------+
|Amazon Tap - Alexa-Enabled Portable Bluetooth Speaker                                    |358      |7       |51.142857142857146|
|Echo Dot (2nd Generation) - Black                                                        |12       |1       |12.0              |
|Amazon Fire TV                                                                           |26       |4       |6.5               |
|Amazon Fire TV Game Controller                                                           |6        |1       |6.0               |
|Amazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders

## Longest Review Texts per Category

This query finds the longest review text in each category along with its title and length.


In [10]:
from pyspark.sql.functions import col, length, row_number
from pyspark.sql.window import Window

# Create window specification
windowSpec = Window.partitionBy("primary_category").orderBy(col("review_length").desc())

# Find longest review per category
longest_reviews = df_clean.withColumn("review_length", length(col("reviews_text"))) \
    .withColumn("rank", row_number().over(windowSpec)) \
    .filter(col("rank") == 1) \
    .select("primary_category", "reviews_title", "reviews_text", "review_length") \
    .orderBy(col("review_length").desc())

# Show results
longest_reviews.show(truncate=True)



+--------------------+--------------------+--------------------+-------------+
|    primary_category|       reviews_title|        reviews_text|review_length|
+--------------------+--------------------+--------------------+-------------+
|          Categories|This box is a GAM...|I am not a casual...|        19739|
|      Amazon Devices|Excellent 3rd-gen...|This is the middl...|        18667|
|Amazon Devices & ...|Great range, very...|As other reviewer...|         1925|
|         Electronics|Fantastic tablet ...|Let me start by s...|         1778|
|        Kindle Store|Worth the money. ...|The Kindle is my ...|         1672|
|Cell Phones & Acc...|Moshi's screen pr...|I like Moshi's an...|         1379|
+--------------------+--------------------+--------------------+-------------+



## Year-over-Year Growth in Review Counts

This query shows how number of reviews has grown year by year.


In [11]:
from pyspark.sql.functions import col, year, to_timestamp, count, lag
from pyspark.sql.window import Window

df_year = df_clean.withColumn(
    "year",
    year(to_timestamp(col("reviews_date")))
)

yearly_counts = df_year.groupBy("year") \
    .agg(count("*").alias("review_count")) \
    .orderBy("year")

windowSpec = Window.orderBy("year")

growth = yearly_counts.withColumn(
    "prev_year_count",
    lag("review_count").over(windowSpec)
)

growth = growth.withColumn(
    "growth_percent",
    ((col("review_count") - col("prev_year_count")) / col("prev_year_count")) * 100
)

growth.show()




+----+------------+---------------+------------------+
|year|review_count|prev_year_count|    growth_percent|
+----+------------+---------------+------------------+
|NULL|         217|           NULL|              NULL|
|2012|           5|            217| -97.6958525345622|
|2013|          24|              5|             380.0|
|2014|         101|             24|320.83333333333337|
|2015|          18|            101|-82.17821782178217|
|2016|         328|             18|1722.2222222222222|
|2017|         484|            328|  47.5609756097561|
+----+------------+---------------+------------------+



## Average Rating by Review Length Buckets

Reviews are divided into:

- Short (<50 chars)  
- Medium (50–200)  
- Long (>200)

Average rating for each bucket is calculated.


In [12]:
from pyspark.sql.functions import when

bucket_df = df_clean.withColumn("length", length("reviews_text"))

bucket_df = bucket_df.withColumn("bucket",
    when(col("length") < 50, "Short")
    .when((col("length") >= 50) & (col("length") <= 200), "Medium")
    .otherwise("Long")
)

bucket_df.groupBy("bucket") \
         .agg(avg("reviews_rating_int").alias("avg_rating")) \
         .show()


+------+-----------------+
|bucket|       avg_rating|
+------+-----------------+
|Medium|4.480984340044743|
|  Long|4.270503597122302|
| Short|4.571428571428571|
+------+-----------------+



## Products with Declining Ratings Over Time

This query identifies products whose ratings have dropped the most over time by comparing first and last monthly averages.


In [13]:
from pyspark.sql.functions import col, to_timestamp, date_format, avg, first, last
from pyspark.sql.window import Window

df_month = df_clean.withColumn(
    "month",
    date_format(to_timestamp(col("reviews_date")), "yyyy-MM")
)

monthly_avg = df_month.groupBy("name", "month") \
    .agg(avg("reviews_rating_int").alias("avg_rating")) \
    .orderBy("name", "month")


windowSpec = Window.partitionBy("name").orderBy("month") \
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

decline_df = monthly_avg.withColumn(
    "first_avg",
    first("avg_rating").over(windowSpec)
).withColumn(
    "last_avg",
    last("avg_rating").over(windowSpec)
)

decline_df = decline_df.withColumn(
    "drop",
    col("first_avg") - col("last_avg")
)
result = decline_df.select("name", "drop") \
    .distinct() \
    .orderBy(col("drop").desc())

result.show(10, truncate=False)


+-----------------------------------------------------------------------------------------+------------------+
|name                                                                                     |drop              |
+-----------------------------------------------------------------------------------------+------------------+
|Replacement Remote for Amazon Fire TV Stick                                              |4.0               |
|Alexa Voice Remote for Amazon Echo and Echo Dot                                          |3.0               |
|Kindle Fire HDX 8.9"                                                                     |2.5               |
|Kindle for Kids Bundle with the latest Kindle E-reader                                   |2.0               |
|Fire HDX 8.9 Tablet                                                                      |2.0               |
|All-New Amazon Fire HD 8 Tablet Case (7th Generation                                     |1.833333333333333 |
|

## Performance Optimization

The monthly trend query is computationally heavy.  
Caching is used to improve performance by avoiding recomputation.


In [14]:
import time

start = time.time()
trend.groupBy("primary_category", "month").agg(avg("reviews_rating_int")).show()
end = time.time()

print("Execution Time Before Optimization:", end-start)

trend.cache()

start = time.time()
trend.groupBy("primary_category", "month").agg(avg("reviews_rating_int")).show()
end = time.time()

print("Execution Time After Optimization:", end-start)


+--------------------+-------+-----------------------+
|    primary_category|  month|avg(reviews_rating_int)|
+--------------------+-------+-----------------------+
|Amazon Devices & ...|2016-03|     1.6666666666666667|
|Amazon Devices & ...|2017-01|                    1.0|
|      Amazon Devices|2012-09|                    4.5|
|      Amazon Devices|2017-04|      4.604651162790698|
|      Amazon Devices|2016-04|                    4.0|
|Amazon Devices & ...|2016-07|                    2.0|
|Cell Phones & Acc...|2014-08|                    5.0|
|Amazon Devices & ...|2016-04|                    3.0|
|      Amazon Devices|2013-10|                 4.3125|
|      Amazon Devices|2014-07|                    5.0|
|         Electronics|2016-09|                    5.0|
|      Amazon Devices|2012-10|                    4.0|
|      Amazon Devices|2016-09|      4.544117647058823|
|Amazon Devices & ...|2016-12|                    1.0|
|      Amazon Devices|2013-12|                    5.0|
|Cell Phon

## Conclusion

All analytical queries were successfully implemented using PySpark.

Key achievements:

- Correct loading of complex CSV  
- Data cleansing and transformation  
- Aggregation and trend analysis  
- Performance optimization  
